# Notas — Aula 14: Design Patterns II (Observer, State)

Marco: sensores do robô passam a **avisar sozinhos** quando algo relevante acontece —
bate numa parede, a bateria cruza o limiar crítico — sem que ninguém precise ficar
perguntando em loop. O robô também ganha **modos de operação** —
`ModoExplorando`/`ModoCarregando` — que mudam seu comportamento inteiro, trocados
automaticamente quando um observador avisa que a bateria está crítica.


In [1]:
from enum import Enum

LADO_GRADE = 10

class Direcao(Enum):
    LESTE = (1, 0)
    NORTE = (0, 1)
    OESTE = (-1, 0)
    SUL = (0, -1)


## Observer: `Sujeito`/`Observador` — mecânica pura

Um `Sujeito` guarda uma lista de observadores e, quando `notificar` é chamado, avisa
todo mundo da lista chamando `atualizar` em cada um. Nenhum dos dois lados precisa
herdar de uma classe em comum além do contrato mínimo — só precisa ter o método certo
(duck typing, como `EstrategiaPadrao`/`EstrategiaZigzag` na Aula 9). No robô, isso é o
que vai deixar sensores avisarem sozinhos quando algo acontece.


In [2]:
class Sujeito:
    def __init__(self):
        self._observadores = []

    def adicionar_observador(self, observador):
        self._observadores.append(observador)

    def remover_observador(self, observador):
        self._observadores.remove(observador)

    def notificar(self, **dados):
        for obs in self._observadores:
            obs.atualizar(**dados)


class Observador:
    def atualizar(self, **dados):
        raise NotImplementedError


class PrintObservador(Observador):
    def __init__(self, nome):
        self.nome = nome

    def atualizar(self, **dados):
        print(f"{self.nome} recebeu: {dados}")


s = Sujeito()
s.adicionar_observador(PrintObservador("Obs1"))
s.adicionar_observador(PrintObservador("Obs2"))
s.notificar(evento="teste", valor=42)


Obs1 recebeu: {'evento': 'teste', 'valor': 42}
Obs2 recebeu: {'evento': 'teste', 'valor': 42}


### Sua vez

Complete `Logger.atualizar`: em vez de imprimir, guarde cada `dados` recebido (um
dicionário) numa lista `self.eventos`.

*Dica: `self.eventos.append(dados)`.*


In [3]:
class Logger(Observador):
    def __init__(self):
        self.eventos = []

    def atualizar(self, **dados):
        # TODO: guarde dados em self.eventos
        ...


log = Logger()
s2 = Sujeito()
s2.adicionar_observador(log)
s2.notificar(evento="primeiro")
s2.notificar(evento="segundo")
print(log.eventos)


[]


## Robô observável: obstáculo e bateria crítica avisando sozinhos

`Robo` não herda de `Sujeito` — o mesmo debate composição-vs-herança da Aula 5. Em vez
disso, o protocolo (`adicionar_observador`/`notificar`) entra direto como métodos do
`Robo`: ele **tem** um mecanismo de notificação, não **é** um sujeito. `avancar()`
notifica `"obstaculo"` ao bater numa parede; `gastar_bateria()` notifica
`"bateria_critica"` só na hora em que a bateria **cruza** o limiar de 20% — não a cada
chamada já em estado crítico.


In [4]:
class Robo:
    LADO_GRADE = 10

    def __init__(self, nome, x=0, y=0, direcao=Direcao.LESTE, obstaculos=None, bateria=100):
        self.nome = nome
        self.x = x
        self.y = y
        self.direcao = direcao
        self.obstaculos = obstaculos if obstaculos is not None else {}
        self.bateria = bateria
        self._observadores = []

    def adicionar_observador(self, obs):
        self._observadores.append(obs)

    def notificar(self, evento, **dados):
        dados.setdefault("robo", self)
        for obs in self._observadores:
            obs.atualizar(evento, **dados)

    @property
    def bateria_critica(self):
        return self.bateria <= 20

    def sensor_frente(self):
        dx, dy = self.direcao.value
        nx, ny = self.x + dx, self.y + dy
        return (0 <= nx < Robo.LADO_GRADE and 0 <= ny < Robo.LADO_GRADE
                and (nx, ny) not in self.obstaculos)

    def avancar(self):
        if self.sensor_frente():
            dx, dy = self.direcao.value
            self.x += dx
            self.y += dy
            return True
        self.notificar("obstaculo", posicao=(self.x, self.y))
        return False

    def gastar_bateria(self, quantidade):
        estava_critica = self.bateria_critica
        self.bateria = max(0, self.bateria - quantidade)
        if self.bateria_critica and not estava_critica:
            self.notificar("bateria_critica", nivel=self.bateria)


class AlertaBateria(Observador):
    def atualizar(self, evento, **dados):
        if evento == "bateria_critica":
            print(f"[ALERTA] bateria crítica: {dados['nivel']}%")


class RegistroEventos(Observador):
    def __init__(self):
        self.eventos = []

    def atualizar(self, evento, **dados):
        self.eventos.append((evento, dados))


robo1 = Robo("Wall-E", x=9, y=0)
robo1.adicionar_observador(AlertaBateria())
registro = RegistroEventos()
robo1.adicionar_observador(registro)

robo1.avancar()            # bate na parede leste
robo1.gastar_bateria(85)   # 100 -> 15, cruza o limiar

print([e for e, d in registro.eventos])


[ALERTA] bateria crítica: 15%
['obstaculo', 'bateria_critica']


### Sua vez

Complete `RegistroEventosComFiltro.eventos_do_tipo(evento)`: devolve só os itens de
`self.eventos` (lista de tuplas `(evento, dados)`) cujo primeiro elemento é igual a
`evento`.

*Dica: `resultado.extend([(e, d) for e, d in self.eventos if e == evento])`.*


In [5]:
class RegistroEventosComFiltro(RegistroEventos):
    def eventos_do_tipo(self, evento):
        resultado = []
        # TODO: preencha resultado com os itens de self.eventos cujo evento é igual ao pedido
        ...
        return resultado


robo2 = Robo("Bender", x=9, y=0)
filtro = RegistroEventosComFiltro()
robo2.adicionar_observador(filtro)
robo2.avancar()
robo2.gastar_bateria(85)
print([e for e, d in filtro.eventos_do_tipo("bateria_critica")])


[]


## State: cada modo vira uma classe

Sem padrão nenhum, cada método sensível a modo teria que checar
`if self.modo == "explorando": ... elif ...` toda vez — repetido em todo lugar que
importa, fácil esquecer de atualizar um deles quando um modo novo aparece (o mesmo
cheiro de código duplicado de atributos redundantes na Aula 7). O **State Pattern**
resolve dando a cada modo sua própria classe, com seu próprio comportamento; `Robo`
guarda **qual objeto de modo** está ativo e delega — a mesma ideia de `self.estrategia`
na Aula 9, só que agora o que muda é "o que o robô pode fazer agora", não "como
navegar".


In [6]:
class ModoOperacao:
    def mover(self, robo):
        raise NotImplementedError


class ModoExplorando(ModoOperacao):
    def mover(self, robo):
        return robo.avancar()


Robo.mover = lambda self: self.modo.mover(self)

robo3 = Robo("Optimus", x=5, y=5)
robo3.modo = ModoExplorando()
robo3.mover()
print(robo3.x, robo3.y)


6 5


### Sua vez

Complete `ModoCarregando.carregar`: soma 30 na bateria do robô (sem passar de 100);
se a bateria chegar a 100, troque `robo.modo` para uma nova `ModoExplorando()`.

*Dica: `robo.bateria = min(100, robo.bateria + 30)`, depois cheque
`if robo.bateria >= 100: robo.modo = ModoExplorando()`.*


In [7]:
class ModoCarregando(ModoOperacao):
    def mover(self, robo):
        print(f"{robo.nome} está carregando, não pode se mover.")
        return False

    def carregar(self, robo):
        # TODO: some 30 na bateria (sem passar de 100); se chegar a 100,
        # troque robo.modo para uma nova ModoExplorando()
        ...


Robo.tick = lambda self: self.modo.carregar(self) if isinstance(self.modo, ModoCarregando) else None

robo4 = Robo("Wall-E", bateria=15)
robo4.modo = ModoCarregando()
robo4.mover()
for _ in range(3):
    robo4.tick()
print(type(robo4.modo).__name__, robo4.bateria)


Wall-E está carregando, não pode se mover.
ModoCarregando 15


## Observer + State conectados: o robô reage sozinho

Até agora quem troca `robo.modo` é o próprio `tick()`, checando a bateria por dentro.
E se, em vez disso, um **observador** disparasse a troca, reagindo à notificação de
bateria crítica — sem ninguém chamar `tick()` nem checar nada manualmente?


In [8]:
class MonitorBateria(Observador):
    def atualizar(self, evento, **dados):
        if evento == "bateria_critica":
            dados["robo"].modo = ModoCarregando()


robo5 = Robo("Bender", x=9, y=0)
robo5.modo = ModoExplorando()
robo5.adicionar_observador(MonitorBateria())
print(type(robo5.modo).__name__)
robo5.gastar_bateria(85)
print(type(robo5.modo).__name__)
robo5.mover()


ModoExplorando
ModoCarregando
Bender está carregando, não pode se mover.


False

## Para aprofundar

- Observer Pattern — Refactoring.Guru: https://refactoring.guru/design-patterns/observer
- State Pattern — Refactoring.Guru: https://refactoring.guru/design-patterns/state
- `unittest.mock` (spies/observadores de teste) — documentação oficial: https://docs.python.org/3/library/unittest.mock.html
- Biblioteca `transitions` (máquina de estados pronta para Python) — GitHub: https://github.com/pytransitions/transitions
- Classificação de padrões (criacional/estrutural/comportamental), Gang of Four — Refactoring.Guru: https://refactoring.guru/design-patterns/catalog
